In [26]:
import requests
import pandas as pd
import numpy as np
from io import StringIO
import time, random

### 1. Free Agent Contract

In [68]:
def Free_Agent_Scrap(year):
    url = f'https://www.spotrac.com/nba/free-agents/signed/_/year/{year}'
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
    data = pd.read_html(StringIO(response.text))[0]
    df = data.iloc[:,[0,2,3,4,5]].copy()
    df.columns = ['From','To','Player','POS','YRS']
    df['is_retained'] = np.where(df['From'] == df['To'], 1, 0)
    df['year'] = year
    df = df.iloc[:,2:]
    return df

In [69]:
all_data = []

for year in range(2017, 2025):
    print(f'Scraping data for year: {year}')
    Free_Agent_data = Free_Agent_Scrap(year)
    all_data.append(Free_Agent_data)
    time.sleep(random.uniform(1, 3))
print('Data_scrap_complete')

FAC_data = pd.concat(all_data)

Scraping data for year: 2017
Scraping data for year: 2018
Scraping data for year: 2019
Scraping data for year: 2020
Scraping data for year: 2021
Scraping data for year: 2022
Scraping data for year: 2023
Scraping data for year: 2024
Data_scrap_complete


### 2. Contract Extensions

In [71]:
def Extension_Scrap(year):
    url = f'https://www.spotrac.com/nba/contracts/extensions/_/year/{year}'
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
    data = pd.read_html(StringIO(response.text))[0]
    df = data.iloc[:,[1,2,5]].copy()
    df.columns = ['Player','POS','YRS']
    df['year'] = year
    df['is_retained'] = 1
    return df

In [77]:
all_data = []

for year in range(2017, 2025):
    print(f'Scraping data for year: {year}')
    Extension_data = Extension_Scrap(year)
    all_data.append(Extension_data)
    time.sleep(random.uniform(1, 3))
print('Data_scrap_complete')

Ext_data = pd.concat(all_data)

Scraping data for year: 2017
Scraping data for year: 2018
Scraping data for year: 2019
Scraping data for year: 2020
Scraping data for year: 2021
Scraping data for year: 2022
Scraping data for year: 2023
Scraping data for year: 2024
Data_scrap_complete


### 3.NBA Cap% of League Cap Rankings

In [ ]:
def Extension_Scrap(year):
    url = f'https://www.spotrac.com/nba/contracts/extensions/_/year/{year}'
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
    data = pd.read_html(StringIO(response.text))[0]
    df = data.iloc[:,[1,2,5]].copy()
    df.columns = ['Player','POS','YRS']
    df['year'] = year
    return df

In [84]:
import requests
from bs4 import BeautifulSoup
import pandas as pd


def Cap_Pct_Scrap(year):
    url = f'https://www.spotrac.com/nba/rankings/player/_/year/{year}/sort/cap_total_league_pct'
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')

    # 1. 精準鎖定母容器 <ul>
    ul_container = soup.find('ul', class_='list-group mb-4 not-premium')

    if ul_container:
        list_items = ul_container.find_all('li')
        data_list = []

        for item in list_items:
            try:
                # 2. 抓取名字：精準鎖定帶有 class="link" 的 <a> 標籤
                player_tag = item.find('a', class_='link') 
                player_name = player_tag.text.strip() if player_tag else "Unknown"
                
                # 3. 抓取位置：鎖定名字下方的 <small class="mt-0 d-block"> 標籤
                info_tag = item.find('small', class_='mt-0 d-block')
                if info_tag:
                    info_text = info_tag.text.strip()  # 例如得到 "GSW, PG"
                    # 利用逗號切割字串，並取後半段的位置文字，再做一次 strip() 清除空格
                    position = info_text.split(',')[-1].strip() if ',' in info_text else "Unknown"
                else:
                    position = "Unknown"
                
                # 4. 抓取 % 數
                cap_pct_str = item.find_all('span')[-1].text.strip() 

                if '%' in cap_pct_str:
                    cap_pct = float(cap_pct_str.replace('%', '')) / 100
                    
                    data_list.append({
                        'Player': player_name,
                        'POS': position,  # 新增位置欄位
                        'Cap_Pct': cap_pct
                    })
            except Exception as e:
                continue
        # 4. 轉換為 DataFrame
        df = pd.DataFrame(data_list)
        df['year'] = year
        return df
    else:
        print("找不到指定的 <ul> 母容器，請檢查 class 名稱是否正確。")
        return pd.DataFrame()  # 回傳空的 DataFrame

In [85]:
df_all = []

for year in range(2017, 2025):
    print('Scraping data for year:', year)
    df = Cap_Pct_Scrap(year)
    df_all.append(df)
    time.sleep(random.uniform(1, 3))

print('Data_scrap_complete')
Cap_pct_df = pd.concat(df_all)

Scraping data for year: 2017
Scraping data for year: 2018
Scraping data for year: 2019
Scraping data for year: 2020
Scraping data for year: 2021
Scraping data for year: 2022
Scraping data for year: 2023
Scraping data for year: 2024
Data_scrap_complete


### Merge 

In [73]:
FAC_data

,Player,POS,YRS,is_retained,year
0,Stephen Curry,PG,5,1,2017
1,Blake Griffin,PF,5,1,2017
2,Gordon Hayward,SF,4,0,2017
3,Jrue Holiday,PG,5,1,2017
4,Otto Porter Jr.,SF,4,0,2017
...,...,...,...,...,...
151,Quenton Jackson,G,1,1,2024
152,Jacob Toppin,SF,1,0,2024
153,Tristan Vukcevic,C,1,1,2024
154,Kai Jones,PF,1,0,2024


In [78]:
Ext_data

,Player,POS,YRS,year,is_retained
0,C.J. McCollum,SG,4,2017,1
1,Rudy Gobert,C,4,2017,1
2,Giannis Antetokounmpo,PF,4,2017,1
3,Steven Adams,C,4,2017,1
4,Victor Oladipo,SG,4,2017,1
...,...,...,...,...,...
28,Kelly Olynyk,PF,2,2024,1
29,Richaun Holmes,C,2,2024,1
30,Mike Conley,PG,2,2024,1
31,John Konchar,SG,3,2024,1


In [86]:
Cap_pct_df

,Player,POS,Cap_Pct,year
0,Stephen Curry,PG,0.3500,2017
1,LeBron James,SF,0.3359,2017
2,Paul Millsap,PF,0.3105,2017
3,Blake Griffin,PF,0.3000,2017
4,Gordon Hayward,SF,0.3000,2017
...,...,...,...,...
445,Terence Davis,SG,0.0005,2024
446,Elfrid Payton,PG,0.0004,2024
447,Brandon Williams,PG,0.0003,2024
448,J.D. Davison,PG,0.0001,2024


In [100]:
df_contracts = pd.concat([FAC_data, Ext_data], axis=0).drop(columns=['POS'])
df_all = pd.merge(df_contracts, Cap_pct_df, on=['Player','year'], how='left').drop(columns=['POS'])
df_all.isna().sum()

Player           0
YRS              0
is_retained      0
year             0
Cap_Pct        251
dtype: int64

In [101]:
df_all[df_all['Cap_Pct'].isna()]

,Player,YRS,is_retained,year,Cap_Pct
60,Jeff Withey,2,0,2017,NaN
67,Diamond Stone,2,0,2017,NaN
72,Tony Allen,1,0,2017,NaN
84,Anthony Morrow,1,0,2017,NaN
90,Damien Wilkins,1,0,2017,NaN
...,...,...,...,...,...
1066,Jacob Toppin,1,0,2024,NaN
1067,Tristan Vukcevic,1,1,2024,NaN
1068,Kai Jones,1,0,2024,NaN
1069,Collin Gillespie,1,0,2024,NaN
